# Notebook 02 — Feature Engineering

**Question:** which quantities actually separate a healthy bearing from a failing one, and
why should they?

The model is not what makes this work — an Isolation Forest over raw RMS is a few lines.
What decides whether it detects anything early is which quantities it is given. This
notebook builds two sets, and only the second currently reaches the detector:

- **From a raw waveform** (`FeatureExtractor`, demonstrated below on a synthetic signal):
  time-domain shape factors — crest, clearance, impulse — and frequency-domain features
  from a Welch PSD: spectral entropy, band energy at BPFO / BPFI / BSF harmonics, and the
  high-frequency ratio.
- **From the per-file summary table** (`enrich_processed`, run below on the real data):
  rolling statistics and rate of change over the RMS and kurtosis channels.

The first set needs a raw waveform and the pipeline reads the summary table, so nothing in
it reaches the model. Joining the two is the next piece of work; until then this notebook
shows what those features do, not what they contribute.

Not all of them move in the direction intuition suggests. Where they do not, the direction
is stated next to the feature and pinned by a test.

---


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.fft import fft, fftfreq

from nasa_bearing_anomaly.config import FIGURES_DIR
from nasa_bearing_anomaly.features import FeatureExtractor, enrich_processed

print("Imports OK")

## 1. Understanding the Features: A Physics Tour

### 1.1 Time-Domain Features

In [ ]:
# Simulate healthy vs faulty bearing signals to demonstrate features
np.random.seed(42)
fs = 20000
t = np.linspace(0, 1, fs)

# Healthy: Gaussian noise + harmonic
healthy = np.random.normal(0, 0.1, fs) + 0.05 * np.sin(2 * np.pi * 33.33 * t)

# Faulty: Add periodic impulses at BPFO = 236 Hz
bpfo = 236.0
impulse_times = np.arange(0, 1, 1 / bpfo)
faulty = healthy.copy()
for it in impulse_times:
    idx = int(it * fs)
    if idx < fs - 100:
        faulty[idx : idx + 50] += np.random.exponential(0.8) * np.exp(-np.linspace(0, 5, 50))

extractor = FeatureExtractor(fs=fs)

print("═" * 55)
print(f"{'Feature':<20} {'Healthy':>15} {'Faulty':>15}")
print("═" * 55)
h_feat = extractor.extract_time_domain(healthy)
f_feat = extractor.extract_time_domain(faulty)
for feat in h_feat:
    print(f"{feat:<20} {h_feat[feat]:>15.4f} {f_feat[feat]:>15.4f}")

In [ ]:
# Visualize the simulated signals
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

t_plot = t[:1000]  # first 50ms

# Time domain
axes[0, 0].plot(t_plot * 1000, healthy[:1000], color="#3fb950", linewidth=0.8)
axes[0, 0].set_title("Healthy Signal — Time Domain", fontweight="bold")
axes[0, 0].set_xlabel("Time (ms)")
axes[0, 0].set_ylabel("Amplitude (g)")

axes[0, 1].plot(t_plot * 1000, faulty[:1000], color="#f85149", linewidth=0.8)
axes[0, 1].set_title("Faulty Signal — Time Domain (impulses visible)", fontweight="bold")
axes[0, 1].set_xlabel("Time (ms)")
axes[0, 1].set_ylabel("Amplitude (g)")


# Frequency domain (FFT)
def plot_fft(ax, x, fs, color, title):
    N = len(x)
    freqs = fftfreq(N, 1 / fs)[: N // 2]
    mag = np.abs(fft(x))[: N // 2] * 2 / N
    ax.semilogy(freqs, mag + 1e-6, color=color, linewidth=0.5, alpha=0.8)
    ax.set_xlim(0, 2000)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude")
    ax.grid(True, alpha=0.3)


plot_fft(axes[1, 0], healthy, fs, "#3fb950", "Healthy — FFT Spectrum")
plot_fft(axes[1, 1], faulty, fs, "#f85149", "Faulty — FFT (BPFO harmonics appear)")

# Mark BPFO and harmonics
for h in range(1, 4):
    axes[1, 1].axvline(x=bpfo * h, color="#d29922", linestyle="--", alpha=0.7, linewidth=1)
    axes[1, 1].text(bpfo * h + 5, 0.001, f"{h}×BPFO", color="#d29922", fontsize=7)

plt.suptitle("Simulated Healthy vs Faulty Bearing Signal", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_signal_demo.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.2 Frequency-Domain Features

**Spectral Entropy** measures how evenly energy is spread across frequency:
- Healthy: broadband, near-Gaussian noise — energy spread evenly → **high** entropy
- Faulty: energy concentrates at BPFO/BPFI harmonics — sharp peaks → **lower** entropy

This runs opposite to the intuition that damage means more chaos. The impulses a
defect produces are *periodic*, so they add structure to the spectrum rather than
removing it. The direction is verified against the synthetic signals below and
asserted in `tests/test_features.py`.

Because entropy falls for a reason that other things can also cause, read it
alongside defect-frequency band energy rather than on its own.

In [ ]:
print("Frequency-Domain Features:")
print("─" * 50)
h_freq = extractor.extract_frequency_domain(healthy)
f_freq = extractor.extract_frequency_domain(faulty)
for feat in h_freq:
    print(f"{feat:<25} {h_freq[feat]:>10.4f}  {f_freq[feat]:>10.4f}")
print()
print("Note: bpfo_energy should be much higher in the faulty signal!")

## 2. Apply Feature Engineering to All Tests

In [ ]:
enriched_dfs = {}
for test_id in [1, 2, 3]:
    print(f"\nTest {test_id}:")
    enriched_dfs[test_id] = enrich_processed(test_id)

In [ ]:
# Show the new enriched columns
df = enriched_dfs[1]
new_cols = [c for c in df.columns if "roll" in c or "diff" in c]
print(f"Rolling + Rate-of-change features added: {len(new_cols)}")
print("Example columns:")
for c in new_cols[:15]:
    print(f"  {c}")

## 3. Rolling Features — The Trend Signal

A rolling mean smooths noise and reveals the degradation *trend*.
Rate-of-change detects *acceleration* in degradation — often the earliest warning.

In [ ]:
# Visualize rolling features for Test 1 (Bearing3)
df1 = enriched_dfs[1]
raw_col = "Bearing3_ch1_rms"
roll_col = "Bearing3_ch1_rms_roll_mean_50"
diff_col = "Bearing3_ch1_rms_diff_abs"

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

if raw_col in df1.columns:
    ax1.plot(df1.index, df1[raw_col], color="#58a6ff", linewidth=0.7, label="Raw RMS", alpha=0.8)
if roll_col in df1.columns:
    ax1.plot(df1.index, df1[roll_col], color="#f85149", linewidth=1.5, label="Rolling Mean (50)")
ax1.set_title("Test 1 — Bearing3 RMS: Raw vs Rolling Mean", fontweight="bold")
ax1.set_ylabel("RMS (g)")
ax1.legend()
ax1.grid(True, alpha=0.4)

kurt_col = "Bearing3_ch1_kurt"
roll_kurt = "Bearing3_ch1_kurt_roll_mean_50"
if kurt_col in df1.columns:
    ax2.plot(df1.index, df1[kurt_col], color="#d29922", linewidth=0.7, alpha=0.7, label="Kurtosis")
if roll_kurt in df1.columns:
    ax2.plot(df1.index, df1[roll_kurt], color="#bc8cff", linewidth=1.5, label="Rolling Kurt (50)")
ax2.axhline(y=5, color="#f85149", linestyle="--", alpha=0.6)
ax2.set_title("Kurtosis: Raw vs Rolling", fontweight="bold")
ax2.set_ylabel("Kurtosis")
ax2.legend()
ax2.grid(True, alpha=0.4)

if diff_col in df1.columns:
    ax3.plot(df1.index, df1[diff_col], color="#3fb950", linewidth=0.7)
    ax3.fill_between(df1.index, df1[diff_col], alpha=0.3, color="#3fb950")
ax3.set_title("Rate of Change in RMS (early warning signal)", fontweight="bold")
ax3.set_xlabel("File Index (Time →)")
ax3.set_ylabel("|ΔRMS|")
ax3.grid(True, alpha=0.4)

plt.suptitle(
    "Feature Engineering: Rolling & Rate-of-Change Features", fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_rolling_test1.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Feature Importance Preview

Compare feature distributions in early (healthy) vs late (degraded) period.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

features_to_plot = [
    ("Bearing3_ch1_rms", "RMS"),
    ("Bearing3_ch1_kurt", "Kurtosis"),
    ("Bearing3_ch1_std", "Std Dev"),
    ("Bearing3_ch1_max", "Peak Value"),
    ("Bearing3_ch1_skew", "Skewness"),
    ("Bearing3_ch2_rms", "RMS (ch2)"),
]

df1 = enriched_dfs[1]
n = len(df1)
early = df1.iloc[: int(n * 0.4)]
late = df1.iloc[int(n * 0.8) :]

for ax, (col, label) in zip(axes, features_to_plot, strict=True):
    if col in df1.columns:
        ax.hist(
            early[col].dropna(),
            bins=30,
            color="#3fb950",
            alpha=0.7,
            density=True,
            label="Healthy (0-40%)",
        )
        ax.hist(
            late[col].dropna(),
            bins=30,
            color="#f85149",
            alpha=0.7,
            density=True,
            label="Degraded (80-100%)",
        )
        ax.set_title(label, fontweight="bold")
        ax.set_ylabel("Density")
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.suptitle("Test 1 — Feature Distributions: Healthy vs Degraded", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_distributions_test1.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary

**On a synthetic waveform**, `FeatureExtractor` produced 11 time-domain and 9
frequency-domain features, and the healthy/faulty comparison above shows which way each
one moves.

**On the real data**, what was written to `data/processed/test{N}_features.csv` is rolling
mean, standard deviation and maximum over windows of 10 and 50 files, plus first
differences — on the RMS and kurtosis channels only. That table is what notebook 03 scores.

So the frequency-domain features are demonstrated here and absent from the pipeline. The
distributions above separate healthy from degraded on the features that are in it; whether
the defect-frequency features would add to that is an ablation nobody has run.

**Next:** Anomaly detection models → Notebook 03
